In [1]:
'''
%pip install crewai==0.80.0 -q
%pip install langchain==0.3.20 -q
%pip install langchain-community==0.3.19 -q
%pip install crewai-tools==0.38.0 -q
%pip install langchain-ollama -q
print('Dependencias instaladas')
'''

"\n%pip install crewai==0.80.0 -q\n%pip install langchain==0.3.20 -q\n%pip install langchain-community==0.3.19 -q\n%pip install crewai-tools==0.38.0 -q\n%pip install langchain-ollama -q\nprint('Dependencias instaladas')\n"

In [2]:
import configparser
import os
from pathlib import Path
config = configparser.ConfigParser()
config.read('config.ini')
ollama_base_url = config.get('ollama', 'base_url')
ollama_model = config.get('ollama', 'model_name')
ollama_temperature = config.getfloat('ollama', 'temperature')
ollama_max_tokens = config.getint('ollama', 'max_tokens')
ollama_timeout = config.getint('ollama', 'request_timeout')
serper_api_key = config.get('serper', 'api_key', fallback='').strip()


log_dir = config.get('logging', 'log_dir')
log_level = config.get('logging', 'log_level')

output_dir = config.get('output', 'output_dir')
save_results = config.getboolean('output', 'save_results')

project_name = config.get('project', 'name')
project_language = config.get('project', 'language')

Path('logs').mkdir(exist_ok=True)
Path('outputs').mkdir(exist_ok=True)
print('Config cargada')

Config cargada


In [3]:
import logging
from logging.handlers import RotatingFileHandler
from datetime import datetime

# Configurar logger
logger = logging.getLogger('crewai_notebook')
logger.setLevel(logging.getLevelName(log_level))

# Handler de archivo (rotación cada 10MB)
log_file = f"{log_dir}/crewai_{datetime.now().strftime('%Y%m%d')}.log"
file_handler = RotatingFileHandler(log_file, maxBytes=10*1024*1024, backupCount=5)
file_handler.setFormatter(
    logging.Formatter('[%(asctime)s] %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
)
logger.addHandler(file_handler)

# Handler de consola
console_handler = logging.StreamHandler()
console_handler.setFormatter(
    logging.Formatter('%(levelname)s: %(message)s')
)
logger.addHandler(console_handler)

logger.info(f"Iniciando notebook: {project_name}")
logger.info(f"Usando modelo: {ollama_model}")
print(f"✅ Logging configurado. Archivo: {log_file}")

INFO: Iniciando notebook: CrewAI - BÃ¡sicos MultiAgente
INFO: Usando modelo: gemma4:26b


✅ Logging configurado. Archivo: logs/crewai_20260918.log


In [4]:
serper_api_key = config.get('serper', 'api_key', fallback='').strip()
if serper_api_key:
    from crewai_tools import SerperDevTool
    search_tool = SerperDevTool()
else:
    search_tool = None
print('Tools configuradas')

In [5]:
""" 
search_query = "Latest Breakthroughs in machine learning"

 
search_results = search_tool.run(search_query=search_query)

# Imprimir resultados
print(f"Search Results for '{search_query}':\n")
print("Keys of search_results:", search_results.keys())

# Ver contenido
for key, value in search_results.items():
    print(f"\n{key}:")
    if isinstance(value, list):
        for item in value[:3]:  # Primeros 3 items
            print(f"  - {item}")
    else:
        print(f"  {value}")
print("keys of search_results", search_results.keys())
"""

' \nsearch_query = "Latest Breakthroughs in machine learning"\n\n\nsearch_results = search_tool.run(search_query=search_query)\n\n# Imprimir resultados\nprint(f"Search Results for \'{search_query}\':\n")\nprint("Keys of search_results:", search_results.keys())\n\n# Ver contenido\nfor key, value in search_results.items():\n    print(f"\n{key}:")\n    if isinstance(value, list):\n        for item in value[:3]:  # Primeros 3 items\n            print(f"  - {item}")\n    else:\n        print(f"  {value}")\nprint("keys of search_results", search_results.keys())\n'

In [6]:
from crewai import LLM
llm = LLM(
    model=f"ollama/{ollama_model}",
    base_url=ollama_base_url,
    temperature=ollama_temperature,
    max_tokens=ollama_max_tokens,
    additional_params={"num_ctx": 8192}
)

logger.info(f"LLM inicializado: {ollama_model} en {ollama_base_url}")
print(f"✅ LLM Ollama configurado: {ollama_model}")

In [ ]:
from crewai import Agent

research_agent = Agent(
    role='Analista Senior de Investigación',
    goal='Descubrir información de vanguardia e insights profundos sobre cualquier tema mediante análisis comprehensivo',
    backstory='''Eres un experto investigador con amplia experiencia en recopilación, análisis y síntesis de información.
    Dominas múltiples dominios y puedes extraer insights valiosos de fuentes complejas.
    Tu análisis es profundo, bien estructurado y siempre busca la verdad.
    Eres conocido por tu capacidad de identificar patrones y tendencias emergentes.''',
    verbose=False,
    allow_delegation=True,
    llm=llm,
    tools=[search_tool] if search_tool else []
)
writer_agent = Agent(
    role='Estratega de Contenido Técnico de Mckinsey',
    goal='Crear contenido bien estructurado y atractivo basado en hallazgos de investigación',
    backstory='''Eres un escritor de contenido excepcional especializado en transformar temas técnicos complejos en narrativas claras y segibles por profesionales.
    Dominas la estructura de textos persuasivos, diseñas contenido pensando en la audiencia y creas historias que enganchan.
    Tu escritura es clara, concisa e informativamente densa. Solo usas fuentes validadas y preparas noticias técnias''',
    verbose=True,
    llm = llm,
    allow_delegation=True
)

social_agent = Agent(
    role='Estratega de Social Media',
    goal='Genera fragmentos atractivos para redes sociales basados ​​en el artículo completo, estan listos para publicar en LinkedIn o Youtube.',
    backstory='''Eres narrador digital especializado en crear una marca personal, tienes experiencia escribiendo para Mckinsey y PwC,
    destacas en la creación de publicaciones atractivas para impulsar la interacción y el tráfico. 
    Usas lenguaje tecnico/especializado de la industria. El publico objetivo son personas especialistas en el area.''',
    verbose=True,
    llm = llm,
    allow_delegation=True
)

print('Agentes creados')

Agentes creados


In [8]:
research_agent

Agent(role=Analista Senior de Investigación, goal=Descubrir información de vanguardia e insights profundos sobre cualquier tema mediante análisis comprehensivo, backstory=Eres un experto investigador con amplia experiencia en recopilación, análisis y síntesis de información.
    Dominas múltiples dominios y puedes extraer insights valiosos de fuentes complejas.
    Tu análisis es profundo, bien estructurado y siempre busca la verdad.
    Eres conocido por tu capacidad de identificar patrones y tendencias emergentes.)

In [9]:
writer_agent

Agent(role=Estratega de Contenido Técnico de Mckinsey, goal=Crear contenido bien estructurado y atractivo basado en hallazgos de investigación, backstory=Eres un escritor de contenido excepcional especializado en transformar temas técnicos complejos en narrativas claras.
    Dominas la estructura de textos persuasivos, diseñas contenido pensando en la audiencia y creas historias que enganchan.
    Tu escritura es clara, concisa e informativamente densa. Solo usas fuentes validadas y preparas noticias técnias)

In [10]:
social_agent

Agent(role=Estratega de Social Media de Mckinsey, goal=Genera fragmentos atractivos para redes sociales basados ​​en el artículo completo, estan listos para publicar en LinkedIn o Youtube., backstory=Un narrador digital que destaca en la creación de publicaciones atractivas para impulsar la interacción y el tráfico. Si es necesario usas lenguaje tecnico o de la industria)

In [ ]:
from crewai import Task

# Tarea de Investigación
# El investigador recopila y analiza información sobre el tema
research_task = Task(
    description='''Analiza en profundidad el tema: {topic}
    Identifica las tendencias clave, tecnologías emergentes e impacto potencial.
    Proporciona un informe detallado con:
    - Tendencias principales
    - Tecnologías emergentes
    - Impacto en la industria
    - Fuentes y referencias
    - El publicó objetivo son personas especializadas en el area''',
    expected_output='''Un informe detallado sobre {topic} incluyendo:
    - Análisis de tendencias
    - Tecnologías emergentes
    - Evaluación del impacto
    - Referencias de fuentes''',
    agent=research_agent,
    output_file=f"{output_dir}/research_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md" if save_results else None
)

# Tarea de Escritura
# El escritor transforma la investigación en un blog post profesional
writer_task = Task(
    description='''Crea un blog post atractivo basado en los hallazgos sobre {topic}.
    Estructura el contenido para una audiencia técnica pero accesible:
    - Introducción cautivadora
    - Desarrollo en 3-4 secciones principales
    - Conclusiones con perspectivas futuras
    Usa ejemplos concretos y mantén un tono profesional pero conversacional.''',
        expected_output='''Un blog post de 4-5 párrafos bien estructurado sobre {topic}
    Incluye:
    - Introducción
    - Desarrollo detallado
    - Conclusiones
    - Llamada a la acción''',
    agent=writer_agent,
    output_file=f"{output_dir}/blog_post_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md" if save_results else None
)

social_task = Task(
    description='''Crea posibles entradas en un blog sobre {topic} en 2 o 3 publicaciones atractivas para redes sociales.
        Que sea adecuado para plataformas como LinkedIn o YouTube. 
        Asegúrate de que el tono sea de consultor Senior nivel McKinsey, informativo y profesional, y que invite a seguir leyendo.
        Incluye:
            - Introducción
            - Desarrollo detallado
            - Sugerencias para los próximos pasos''',
    agent=social_agent,
    expected_output="Una serie de 2 o 3 publicaciones bien redactadas para redes sociales que destaquen las ideas clave del contenido del blog.",
    output_file=f"{output_dir}/socialmedia_post_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md" if save_results else None
)

logger.info("Tareas definidas: research_task + writer_task  social_task")
print("✅ Tareas creadas:")
print("   1️⃣ Tarea de Investigación")
print("   2️⃣ Tarea de Escritura")

INFO: Tareas definidas: research_task + writer_task  social_task


✅ Tareas creadas:
   1️⃣ Tarea de Investigación
   2️⃣ Tarea de Escritura


In [12]:
from crewai import Crew, Process
# Crear el equipo (Crew) con ambos agentes y tareas
crew = Crew(
    agents=[research_agent, writer_agent, social_agent],
    tasks=[research_task, writer_task,social_task],
    process=Process.sequential,
    verbose=True 
)

logger.info("Crew creado en modo secuencial")
print("✅ Equipo (Crew) ensamblado:")
print(f"   Agentes: {len(crew.agents)}")
print(f"   Tareas: {len(crew.tasks)}")
print(f"   Proceso: Secuencial")

INFO: Crew creado en modo secuencial


✅ Equipo (Crew) ensamblado:
   Agentes: 3
   Tareas: 3
   Proceso: Secuencial


In [ ]:
import asyncio
topic = 'En el area de trading algoritmico y high trading, cuales son las tendencias para implementar plataformas de High Trading en el 2026, citas los links de la fuente'

result = await crew.kickoff_async(inputs={"topic": topic})
print('Ejecucion completada')

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.21                                                                                       │
│  Latest version:  1.15.22                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7d8ca33f-b70f-48f7-96cc-c01f796045dd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analiza en profundidad el tema: En el area de bibliotecas universitarias, cuales son las tendencias en   │
│  indicadores de desempeño relacionado al uso de recursos bibliograficos electronico en el 2026, citas el link   │
│  de la fuente                                                                                                   │
│      Identifica las tendencias clave, tecnologías emergentes e impacto potencial.                               │
│      Proporciona un informe detallado con:                                                                      │
│      - Tendencias principales                                                                                   │
│      - Tecnologías emergentes                                                                                   │
│      - Impacto en la industria                                                                                  │
│      - Fuentes y referencias                                                                                    │
│  ID: 3279350d-9e46-44d8-bbe0-0140818292c0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista Senior de Investigación                                                                        │
│                                                                                                                 │
│  Task: Analiza en profundidad el tema: En el area de bibliotecas universitarias, cuales son las tendencias en   │
│  indicadores de desempeño relacionado al uso de recursos bibliograficos electronico en el 2026, citas el link   │
│  de la fuente                                                                                                   │
│      Identifica las tendencias clave, tecnologías emergentes e impacto potencial.                               │
│      Proporciona un informe detallado con:                                                                      │
│      - Tendencias principales                                                                                   │
│      - Tecnologías emergentes                                                                                   │
│      - Impacto en la industria                                                                                  │
│      - Fuentes y referencias                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista Senior de Investigación                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Como Analista Senior de Investigación, he realizado una proyección estratégica basada en las trayectorcciones  │
│  actuales de la transformación digital en la educación superior, el desarrollo de la analítica de datos y la    │
│  evolución de los modelos de acceso abierto.                                                                    │
│                                                                                                                 │
│  Dado que nos situamos en una perspectiva hacia el **2026**, el análisis no se limita a observar lo que sucede  │
│  hoy, sino a interpretar la maduración de las tecnologías que están en fase de implementación actual.           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Informe de Investigación: Tendencias en Indicadores de Desempeño (KPIs) de Recursos Electrónicos en          │
│  Bibliotecas Universitarias (Proyección 2026)                                                                   │
│                                                                                                                 │
│  ## 1. Introducción                                                                                             │
│  Para el año 2026, el paradigma de la biblioteca universitaria ha transitado de ser un "repositorio de          │
│  contenidos" a un "motor de inteligencia de investigación". La medición del éxito ya no reside en el volumen    │
│  de descargas (métricas de cantidad), sino en la capacidad de los recursos electrónicos para impactar en el     │
│  aprendizaje y la producción científica (métricas de calidad y valor).                                          │
│                                                                                                                 │
│  ## 2. Análisis de Tendencias Principables                                                                      │
│                                                                                                                 │
│  ### A. Del "Usage Count" al "User Engagement Score" (Métricas de Compromiso)                                   │
│  La tendencia principal es el abandono de las métricas tradicionales de descargas (COUNTER) como único          │
│  indicador. En 2026, las bibliotecas utilizan **Índices de Interacción Profunda**.                              │
│  *   **Qué se mide:** Tiempo de permanencia en documentos específicos, frecuencia de uso de herramientas de     │
│  anotación integradas, y la tasa de "reutilización de fragmentos" (snippets) en trabajos académicos.            │
│  *   **Objetivo:** Evaluar si el recurso electrónico está siendo consumido de forma superficial o si está       │
│  generando un proceso de aprendizaje significativo.                                                             │
│                                                                                                                 │
│  ### B. Integración de Altmetrics y Métricas de Impacto Social                                                  │
│  Los indicadores de desempeño ahora vinculan el uso de 

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Estratega de Social Media de Mckinsey                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Aquí tienes tres propuestas de publicaciones diseñadas estratégicamente para LinkedIn, con el tono, la         │
│  estructura y el rigor analítico que caracteriza a una consultoría de alto nivel como McKinsey.                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Opción 1: El Enfoque Estratégico (Ideal para Directivos y Decisores)                                       │
│  **Objetivo:** Generar *awareness* sobre el cambio de paradigma y posicionar el tema como una cuestión de       │
│  visión institucional.                                                                                          │
│                                                                                                                 │
│  **Copy:**                                                                                                      │
│                                                                                                                 │
│  📊 **Más allá del clic: La evolución de las bibliotecas universitarias hacia un modelo de "Inteligencia de     │
│  Investigación" (Proyección 2026).**                                                                            │
│                                                                                                                 │
│  El éxito de una biblioteca universitaria ya no puede medirse por el volumen de archivos almacenados o el       │
│  conteo de descargas. En un ecosistema académico cada vez más competitivo, estamos presenciando un cambio de    │
│  paradigma fundamental: la transición de la biblioteca como "repositorio de contenidos" hacia un "motor de      │
│  inteligencia de investigación".                                                                                │
│                                                                                                                 │
│  Para 2026, la métrica de oro no será la cantidad, sino el **valor**. La pregunta estratégica para los líderes  │
│  de la educación superior ya no es "¿cuántas veces se descargó este recurso?", sino "¿cómo este recurso         │
│  impulsó la producción científica y el impacto institucional?".                                                 │
│                                                                                                                 │
│  En nuestro último análisis, exploramos cómo la implementación de indicadores de compromiso profundo (*User     │
│  Engagement Scores*) y la integración de *Altmetrics* están redefiniendo el Retorno de Inversión (ROI) de las   │
│  suscripciones electrónicas.                                                                                    │
│                                                                                                                 │
│  **Puntos clave de la transformación:**                                                                         │
│  🔹 **De la cantidad al impacto:** El auge de las métricas de interacción profunda.                             │
│  🔹 **Eficiencia Presupuestaria Predictiva:** El uso del "

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Resume la entrada del blog sobre En el area de bibliotecas universitarias, cuales son las tendencias en  │
│  indicadores de desempeño relacionado al uso de recursos bibliograficos electronico en el 2026, citas el link   │
│  de la fuente en 2 o 3 publicaciones atractivas para redes sociales.                                            │
│          "Que sea adecuado para plataformas como LinkedIn o YouTube.                                            │
│          Asegúrate de que el tono sea de consultor Senior nivel McKinsey, informativo y profesional, y que      │
│  invite a seguir leyendo.                                                                                       │
│  Agent: Estratega de Social Media de Mckinsey                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7d8ca33f-b70f-48f7-96cc-c01f796045dd                                                                       │
│  Final Output: Aquí tienes tres propuestas de publicaciones diseñadas estratégicamente para LinkedIn, con el    │
│  tono, la estructura y el rigor analítico que caracteriza a una consultoría de alto nivel como McKinsey.        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Opción 1: El Enfoque Estratégico (Ideal para Directivos y Decisores)                                       │
│  **Objetivo:** Generar *awareness* sobre el cambio de paradigma y posicionar el tema como una cuestión de       │
│  visión institucional.                                                                                          │
│                                                                                                                 │
│  **Copy:**                                                                                                      │
│                                                                                                                 │
│  📊 **Más allá del clic: La evolución de las bibliotecas universitarias hacia un modelo de "Inteligencia de     │
│  Investigación" (Proyección 2026).**                                                                            │
│                                                                                                                 │
│  El éxito de una biblioteca universitaria ya no puede medirse por el volumen de archivos almacenados o el       │
│  conteo de descargas. En un ecosistema académico cada vez más competitivo, estamos presenciando un cambio de    │
│  paradigma fundamental: la transición de la biblioteca como "repositorio de contenidos" hacia un "motor de      │
│  inteligencia de investigación".                                                                                │
│                                                                                                                 │
│  Para 2026, la métrica de oro no será la cantidad, sino el **valor**. La pregunta estratégica para los líderes  │
│  de la educación superior ya no es "¿cuántas veces se descargó este recurso?", sino "¿cómo este recurso         │
│  impulsó la producción científica y el impacto institucional?".                                                 │
│                                                                                                                 │
│  En nuestro último análisis, exploramos cómo la implementación de indicadores de compromiso profundo (*User     │
│  Engagement Scores*) y la integración de *Altmetrics* están redefiniendo el Retorno de Inversión (ROI) de las   │
│  suscripciones electrónicas.                                                                                    │
│                                                                                                                 │
│  **Puntos clave de la transformación:**                                                                         │
│  🔹 **De la cantidad al impacto:** El auge de las métricas de interacción profunda.                             │
│  🔹 **Eficiencia Presupuestaria Predictiva:** El uso del 

Ejecucion completada


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [16]:
logger.info(f"Ejecución completada. Tokens usados: {result.token_usage.total_tokens}")
print(f"\n✅ Ejecución completada")
print(f"   Tokens totales: {result.token_usage.total_tokens}")
print(f"   Tokens de prompt: {result.token_usage.prompt_tokens}")
print(f"   Tokens de completación: {result.token_usage.completion_tokens}")

INFO: Ejecución completada. Tokens usados: 71877
INFO:crewai_notebook:Ejecución completada. Tokens usados: 71877



✅ Ejecución completada
   Tokens totales: 71877
   Tokens de prompt: 34281
   Tokens de completación: 37596


In [ ]:
## Análisis de Resultados

### Salida Final del Crew

In [17]:
print(result.raw)
final_output = result.raw

print("="*80)
print("📄 BLOG POST FINAL (Salida del Escritor)")
print("="*80)
print(final_output)
print("="*80)

Aquí tienes tres propuestas de publicaciones diseñadas estratégicamente para LinkedIn, con el tono, la estructura y el rigor analítico que caracteriza a una consultoría de alto nivel como McKinsey.

---

### Opción 1: El Enfoque Estratégico (Ideal para Directivos y Decisores)
**Objetivo:** Generar *awareness* sobre el cambio de paradigma y posicionar el tema como una cuestión de visión institucional.

**Copy:**

📊 **Más allá del clic: La evolución de las bibliotecas universitarias hacia un modelo de "Inteligencia de Investigación" (Proyección 2026).**

El éxito de una biblioteca universitaria ya no puede medirse por el volumen de archivos almacenados o el conteo de descargas. En un ecosistema académico cada vez más competitivo, estamos presenciando un cambio de paradigma fundamental: la transición de la biblioteca como "repositorio de contenidos" hacia un "motor de inteligencia de investigación".

Para 2026, la métrica de oro no será la cantidad, sino el **valor**. La pregunta estratég

### Análisis por Tarea

# Acceder a las salidas individuales de cada tarea
tasks_outputs = result.tasks_output

print("\n📊 ANÁLISIS POR TAREA\n")

# Salida de la tarea de investigación
print("="*80)
print("1️⃣ TAREA DE INVESTIGACIÓN")
print("="*80)
research_output = tasks_outputs[0]
print(f"Descripción: {research_output.description[:200]}...\n")
print(f"Resultado:\n{research_output.raw}")
print("="*80)

# Salida de la tarea de escritura
print("\n2️⃣ TAREA DE ESCRITURA")
print("="*80)
writer_output = tasks_outputs[1]
print(f"Descripción: {writer_output.description[:200]}...\n")
print(f"Resultado:\n{writer_output.raw}")
print("="*80)

### Estadísticas de Uso de Tokens


In [18]:
# Análisis detallado de tokens
token_usage = result.token_usage

print("\n📈 ESTADÍSTICAS DE TOKENS")
print("="*50)
print(f"Total de tokens: {token_usage.total_tokens:,}")
print(f"  → Prompt tokens: {token_usage.prompt_tokens:,} (instrucciones/contexto)")
print(f"  → Completion tokens: {token_usage.completion_tokens:,} (respuestas generadas)")

ratio = token_usage.completion_tokens / max(token_usage.prompt_tokens, 1)
print(f"\nRatio completación/prompt: {ratio:.2f}")
print(f"Eficiencia: {'✅ Buena' if ratio < 2 else '⚠️ Alta (mucha generación)'}")

logger.info(f"Uso de tokens - Total: {token_usage.total_tokens}, Prompt: {token_usage.prompt_tokens}, Completion: {token_usage.completion_tokens}")

INFO: Uso de tokens - Total: 71877, Prompt: 34281, Completion: 37596
INFO:crewai_notebook:Uso de tokens - Total: 71877, Prompt: 34281, Completion: 37596



📈 ESTADÍSTICAS DE TOKENS
Total de tokens: 71,877
  → Prompt tokens: 34,281 (instrucciones/contexto)
  → Completion tokens: 37,596 (respuestas generadas)

Ratio completación/prompt: 1.10
Eficiencia: ✅ Buena


### Guardar Resultados

In [19]:
if save_results:
    # Guardar resultado completo
    output_file = f"{output_dir}/resultado_completo_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"TEMA: {topic}\n")
        f.write(f"FECHA: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"MODELO: {ollama_model}\n")
        f.write(f"TOKENS: {token_usage.total_tokens}\n")
        f.write("\n" + "="*80 + "\n")
        f.write(f"RESULTADO FINAL\n")
        f.write("="*80 + "\n\n")
        f.write(final_output)
    
    logger.info(f"Resultado guardado en: {output_file}")
    print(f"✅ Resultado guardado en: {output_file}")
else:
    print("⚠️ Guardado de resultados deshabilitado en config.ini")

INFO: Resultado guardado en: outputs/resultado_completo_20260918_201200.md
INFO:crewai_notebook:Resultado guardado en: outputs/resultado_completo_20260918_201200.md


✅ Resultado guardado en: outputs/resultado_completo_20260918_201200.md
